# 문제 1 — 벡터 연산 모듈 (내적·외적·정사영·rank)

로봇의 좌표 변환은 결국 **벡터 연산**의 조합입니다. 이 노트북에서는
내적·사이각·정규화·정사영·반대칭행렬(외적)·평면 법선·rank 를
`np.linalg` 없이 직접 구현하고, 각각을 검증합니다.

완성한 함수는 `src/vectors.py` 로 내보내 문제 2 이후에서 재사용합니다.

> **규약** — 난수는 `np.random.default_rng(42)` 로 고정하고,
> 수치 비교는 부동소수점 오차를 고려해 `np.allclose` / `np.isclose` 로 합니다.
> `np.linalg` 는 **검산용으로만** 쓰며, 쓸 때마다 주석으로 검산임을 밝힙니다.

In [1]:
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.vectors import (angle_between, cross, det, dot, norm, normalize,
                         plane_normal, project, rank, reject, row_echelon, skew)

rng = np.random.default_rng(42)          # 시드 고정
np.set_printoptions(precision=6, suppress=True)


def check(label, condition):
    """검증 셀에서 쓰는 통과/실패 출력 헬퍼."""
    tag = "PASS" if condition else "FAIL"
    print("[" + tag + "] " + label)
    return bool(condition)


print("NumPy", np.__version__)

NumPy 2.5.2


## 1-1. 내적과 사이각

내적의 정의는 두 가지이며 서로 같습니다.

$$\mathbf{a}\cdot\mathbf{b}=\sum_i a_i b_i = |\mathbf{a}||\mathbf{b}|\cos\theta$$

두 번째 식을 $\theta$ 에 대해 풀면 사이각이 나옵니다.
검증하기 쉽도록 **손으로 계산되는 값**을 골랐습니다.

$\mathbf{a}=(3,4,0)$, $\mathbf{b}=(4,3,0)$ 이면
$\mathbf{a}\cdot\mathbf{b}=12+12+0=24$, $|\mathbf{a}|=|\mathbf{b}|=5$ 이므로
$\cos\theta = 24/25 = 0.96$, $\theta=\arccos 0.96 \approx 16.26^\circ$ 입니다.

In [2]:
a = np.array([3.0, 4.0, 0.0])
b = np.array([4.0, 3.0, 0.0])

d = dot(a, b)
theta_deg = angle_between(a, b, degrees=True)

print("a·b        =", d)
print("|a|, |b|   =", norm(a), norm(b))
print("cos(theta) =", d / (norm(a) * norm(b)))
print("사이각      =", round(theta_deg, 6), "도")

# 손계산 값
hand_dot = 3 * 4 + 4 * 3 + 0 * 0            # = 24
hand_cos = 24 / (5 * 5)                     # = 0.96
hand_deg = np.degrees(np.arccos(hand_cos))  # = 16.260205...
print("\n손계산: a·b =", hand_dot, ", cos =", hand_cos, ", 사이각 =", round(hand_deg, 6), "도")

a·b        = 24.0
|a|, |b|   = 5.0 5.0
cos(theta) = 0.96
사이각      = 16.260205 도

손계산: a·b = 24 , cos = 0.96 , 사이각 = 16.260205 도


In [3]:
# --- 검증 ---
ok = check("내적이 손계산(24)과 일치", np.isclose(d, hand_dot))
ok &= check("사이각이 손계산과 일치", np.isclose(theta_deg, hand_deg))
ok &= check("np.dot 검산과 일치", np.isclose(d, np.dot(a, b)))   # 검산용
ok &= check("수직 벡터의 사이각은 90도",
            np.isclose(angle_between([1, 0, 0], [0, 1, 0]), 90.0))
ok &= check("같은 벡터의 사이각은 0도", np.isclose(angle_between(a, a), 0.0))
print("\n1-1 전체 통과:", ok)

[PASS] 내적이 손계산(24)과 일치
[PASS] 사이각이 손계산과 일치
[PASS] np.dot 검산과 일치
[PASS] 수직 벡터의 사이각은 90도
[PASS] 같은 벡터의 사이각은 0도

1-1 전체 통과: True


## 1-2. 정규화와 영벡터 — 무슨 일이 일어나는가

정규화는 $\hat{\mathbf{v}} = \mathbf{v}/|\mathbf{v}|$ 입니다.
**영벡터를 넣으면 $0/0$** 이라 IEEE 754 규칙에 따라 `nan` 이 나오고,
NumPy 는 `RuntimeWarning: invalid value encountered` 를 띄웁니다.

`nan` 은 예외를 던지지 않고 이후 모든 연산에 조용히 전파되며,
`nan` 과의 비교는 항상 False 라 `assert` 로도 잡히지 않습니다.
로봇 코드에서는 **엉뚱한 자세 명령이 나갈 때까지 아무도 모른다**는 것이 진짜 위험입니다.

→ **선택한 처리 방식: 길이가 `eps=1e-12` 이하면 `ValueError` 를 던진다(fail-fast).**
방향이 수학적으로 정의되지 않는 입력이므로, "0을 반환"이나 "임의의 축 반환"처럼
그럴듯한 값을 지어내는 대신 호출부가 반드시 처리하도록 강제하는 편이 안전합니다.

In [4]:
zero = np.array([0.0, 0.0, 0.0])

# (1) 아무 처리 없이 나눴을 때 실제로 무슨 일이 나는지 기록
with np.errstate(invalid="ignore", divide="ignore"):
    raw = zero / np.sqrt(zero @ zero)
print("보호 없이 나눈 결과 :", raw, "  ->  nan 이 조용히 전파됩니다")
print("nan == nan          :", bool(raw[0] == raw[0]), "(비교가 항상 False 라 assert 로도 못 잡음)")

# (2) 구현한 normalize 의 동작
u = normalize([3.0, 4.0, 0.0])
print("\n정상 벡터 정규화     :", u, " |v| =", norm(u))
try:
    normalize(zero)
except ValueError as e:
    print("영벡터 정규화        : ValueError ->", e)

보호 없이 나눈 결과 : [nan nan nan]   ->  nan 이 조용히 전파됩니다
nan == nan          : False (비교가 항상 False 라 assert 로도 못 잡음)

정상 벡터 정규화     : [0.6 0.8 0. ]  |v| = 1.0
영벡터 정규화        : ValueError -> 영백터는 정규화할 수 없습니다.(|v|=0.000e+00 <= eps=1.000000e-12).방향이 정의되지 않으므로 호출부에서 예외를 처리하세요.


In [5]:
# --- 검증 ---
ok = check("보호 없이 나누면 nan 이 나온다", bool(np.all(np.isnan(raw))))
ok &= check("정규화된 벡터의 길이는 1", np.isclose(norm(u), 1.0))
ok &= check("정규화는 방향을 바꾸지 않는다", np.isclose(angle_between([3.0, 4.0, 0.0], u), 0.0))

try:
    normalize(zero)
    ok &= check("영벡터에서 ValueError 발생", False)
except ValueError:
    ok &= check("영벡터에서 ValueError 발생", True)

v = rng.standard_normal(3)
ok &= check("무작위 벡터도 정규화 후 길이 1", np.isclose(norm(normalize(v)), 1.0))
print("\n1-2 전체 통과:", ok)

[PASS] 보호 없이 나누면 nan 이 나온다
[PASS] 정규화된 벡터의 길이는 1
[PASS] 정규화는 방향을 바꾸지 않는다
[PASS] 영벡터에서 ValueError 발생
[PASS] 무작위 벡터도 정규화 후 길이 1

1-2 전체 통과: True


## 1-3. 정사영 — 수직성과 합 복원

$\mathbf{a}$ 를 $\mathbf{b}$ 방향으로 정사영한 성분은

$$\mathrm{proj}_{\mathbf{b}}(\mathbf{a})=\frac{\mathbf{a}\cdot\mathbf{b}}{\mathbf{b}\cdot\mathbf{b}}\mathbf{b}$$

이고, 남는 성분(reject)은 $\mathbf{a}-\mathrm{proj}_{\mathbf{b}}(\mathbf{a})$ 입니다.
검증할 두 가지는 다음과 같습니다.

1. **수직성**: (남는 성분) $\cdot$ $\mathbf{b} = 0$
2. **합 복원**: $\mathrm{proj} + \mathrm{rej} = \mathbf{a}$

분모가 $|\mathbf{b}|^2$ 이므로 $\mathbf{b}$ 를 미리 정규화할 필요가 없습니다.

In [6]:
a = np.array([2.0, 3.0, 4.0])
b = np.array([1.0, 0.0, 1.0])

p = project(a, b)     # b 방향 성분
r = reject(a, b)      # b 에 수직인 나머지

print("a          =", a)
print("b          =", b)
print("proj_b(a)  =", p, "  (계수 a·b/b·b =", dot(a, b) / dot(b, b), ")")
print("rej_b(a)   =", r)
print("proj + rej =", p + r)
print("rej · b    =", format(dot(r, b), ".3e"))

a          = [2. 3. 4.]
b          = [1. 0. 1.]
proj_b(a)  = [3. 0. 3.]   (계수 a·b/b·b = 3.0 )
rej_b(a)   = [-1.  3.  1.]
proj + rej = [2. 3. 4.]
rej · b    = 0.000e+00


In [7]:
# --- 검증 ---
ok = check("① 남는 성분이 b 와 수직 (rej·b = 0)", np.isclose(dot(r, b), 0.0))
ok &= check("② proj + rej = a (합 복원)", np.allclose(p + r, a))
ok &= check("proj 는 b 와 평행 (외적이 0)", np.allclose(cross(p, b), 0.0))
ok &= check("피타고라스: |a|^2 = |proj|^2 + |rej|^2",
            np.isclose(norm(a) ** 2, norm(p) ** 2 + norm(r) ** 2))

# 무작위 100 쌍에서도 성립하는지
A = rng.standard_normal((100, 3))
B = rng.standard_normal((100, 3))
ok &= check("무작위 100쌍 모두 수직성·합 복원 만족",
            all(np.isclose(dot(reject(x, y), y), 0.0)
                and np.allclose(project(x, y) + reject(x, y), x)
                for x, y in zip(A, B)))
print("\n1-3 전체 통과:", ok)

[PASS] ① 남는 성분이 b 와 수직 (rej·b = 0)
[PASS] ② proj + rej = a (합 복원)
[PASS] proj 는 b 와 평행 (외적이 0)
[PASS] 피타고라스: |a|^2 = |proj|^2 + |rej|^2
[PASS] 무작위 100쌍 모두 수직성·합 복원 만족

1-3 전체 통과: True


## 1-4. 외적을 반대칭행렬 곱으로 — `skew(a)`

외적은 행렬 곱으로 쓸 수 있습니다.

$$\mathbf{a}\times\mathbf{b} = [\mathbf{a}]_\times \mathbf{b},\qquad
[\mathbf{a}]_\times=\begin{bmatrix}0&-a_3&a_2\\a_3&0&-a_1\\-a_2&a_1&0\end{bmatrix}$$

이 형태가 중요한 이유는 **로드리게스 공식(문제 2)과 각속도 → 회전 미분**이
전부 $[\boldsymbol{\omega}]_\times$ 로 표현되기 때문입니다.
반대칭(skew-symmetric)이란 $M^{\mathsf{T}} = -M$ 을 뜻하고,
대각성분이 반드시 0 이라는 성질이 따라옵니다.

In [8]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

S = skew(a)
print("skew(a) =\n", S)
print("\nskew(a) @ b   =", S @ b)
print("np.cross(a,b) =", np.cross(a, b), "  # 검산용")
print("\nskew(a).T =\n", S.T)
print("\n-skew(a)  =\n", -S)

skew(a) =
 [[ 0. -3.  2.]
 [ 3.  0. -1.]
 [-2.  1.  0.]]

skew(a) @ b   = [-3.  6. -3.]
np.cross(a,b) = [-3.  6. -3.]   # 검산용

skew(a).T =
 [[ 0.  3. -2.]
 [-3.  0.  1.]
 [ 2. -1.  0.]]

-skew(a)  =
 [[-0.  3. -2.]
 [-3. -0.  1.]
 [ 2. -1. -0.]]


In [9]:
# --- 검증 ---
ok = check("skew(a) @ b == np.cross(a, b)", np.allclose(S @ b, np.cross(a, b)))
ok &= check("skew(a) 가 반대칭 (S.T == -S)", np.allclose(S.T, -S))
ok &= check("반대칭이므로 대각성분은 모두 0", np.allclose(np.diag(S), 0.0))
ok &= check("skew(a) @ a == 0 (자기 자신과의 외적)", np.allclose(S @ a, 0.0))
ok &= check("반교환성: a x b == -(b x a)", np.allclose(cross(a, b), -cross(b, a)))

# 무작위 200 쌍 일괄 검증
A = rng.standard_normal((200, 3))
B = rng.standard_normal((200, 3))
ok &= check("무작위 200쌍에서 skew 곱 == np.cross",
            all(np.allclose(skew(x) @ y, np.cross(x, y)) for x, y in zip(A, B)))
print("\n1-4 전체 통과:", ok)

[PASS] skew(a) @ b == np.cross(a, b)
[PASS] skew(a) 가 반대칭 (S.T == -S)
[PASS] 반대칭이므로 대각성분은 모두 0
[PASS] skew(a) @ a == 0 (자기 자신과의 외적)
[PASS] 반교환성: a x b == -(b x a)
[PASS] 무작위 200쌍에서 skew 곱 == np.cross

1-4 전체 통과: True


## 1-5. 세 점이 만드는 평면의 단위 법선

세 점 $P_1,P_2,P_3$ 이 주어지면 두 모서리 벡터
$\mathbf{u}=P_2-P_1$, $\mathbf{v}=P_3-P_1$ 의 외적이 평면에 수직입니다.
이를 정규화하면 단위 법선입니다.

세 점이 일직선이면 $\mathbf{u}\times\mathbf{v}=\mathbf{0}$ 이 되어
평면이 하나로 정해지지 않으므로, 이 경우도 예외로 잡습니다.

In [10]:
P1 = np.array([0.0, 0.0, 0.0])
P2 = np.array([1.0, 0.0, 0.0])
P3 = np.array([0.0, 1.0, 0.0])

n = plane_normal(P1, P2, P3)
print("xy 평면 세 점의 단위 법선 :", n, "(기대값: z축 + 방향)")

# 기울어진 평면
Q1 = np.array([1.0, 0.0, 0.0])
Q2 = np.array([0.0, 1.0, 0.0])
Q3 = np.array([0.0, 0.0, 1.0])
nq = plane_normal(Q1, Q2, Q3)
print("x+y+z=1 평면의 단위 법선 :", nq, "(기대값: 각 성분", 1 / np.sqrt(3), ")")

try:
    plane_normal([0, 0, 0], [1, 1, 1], [2, 2, 2])     # 일직선
except ValueError as e:
    print("\n일직선 세 점 : ValueError ->", e)

xy 평면 세 점의 단위 법선 : [0. 0. 1.] (기대값: z축 + 방향)
x+y+z=1 평면의 단위 법선 : [0.57735 0.57735 0.57735] (기대값: 각 성분 0.5773502691896258 )

일직선 세 점 : ValueError -> 세 점이 일직선이라 평면이 하나로 정해지지 않습니다.


In [11]:
# --- 검증 ---
ok = check("법선의 길이가 1", np.isclose(norm(n), 1.0))
ok &= check("법선이 z축과 일치", np.allclose(n, [0.0, 0.0, 1.0]))
ok &= check("법선이 두 모서리 벡터 모두와 수직",
            np.isclose(dot(nq, Q2 - Q1), 0.0) and np.isclose(dot(nq, Q3 - Q1), 0.0))
ok &= check("기울어진 평면의 법선 == (1,1,1)/sqrt(3)", np.allclose(nq, np.ones(3) / np.sqrt(3)))
try:
    plane_normal([0, 0, 0], [1, 1, 1], [2, 2, 2])
    ok &= check("일직선 입력에서 ValueError", False)
except ValueError:
    ok &= check("일직선 입력에서 ValueError", True)
print("\n1-5 전체 통과:", ok)

[PASS] 법선의 길이가 1
[PASS] 법선이 z축과 일치
[PASS] 법선이 두 모서리 벡터 모두와 수직
[PASS] 기울어진 평면의 법선 == (1,1,1)/sqrt(3)
[PASS] 일직선 입력에서 ValueError

1-5 전체 통과: True


## 1-6. rank 와 행렬식 — 왜 3 이 아닌가

세 벡터 $(1,0,1)$, $(0,1,1)$, $(1,1,2)$ 를 행으로 쌓은 행렬의 rank 를 구합니다.
rank 는 **선형독립인 행(또는 열)의 개수**이고, 행 사다리꼴로 만들었을 때
살아남는 피벗의 개수와 같습니다.

눈으로 보면 이미 답이 보입니다.

$$(1,0,1)+(0,1,1)=(1,1,2)$$

즉 **세 번째 벡터가 앞의 두 벡터의 합**이라 새로운 방향을 하나도 더하지 않습니다.
세 벡터는 3차원 공간 전체가 아니라 **평면 하나**만 span 하므로 rank 는 2 입니다.
정사각 행렬에서 rank < n 이면 행렬식은 0 이어야 하므로 두 결과는 서로 일관됩니다.

In [12]:
M = np.array([[1.0, 0.0, 1.0],
              [0.0, 1.0, 1.0],
              [1.0, 1.0, 2.0]])

U, pivots, swaps = row_echelon(M)
print("행 사다리꼴 U =\n", U)
print("\n피벗 열 :", pivots, " -> rank =", len(pivots))
print("rank(M) (직접 구현)      =", rank(M))
print("np.linalg.matrix_rank(M) =", np.linalg.matrix_rank(M), "  # 검산용")
print("\ndet(M) (직접 구현)       =", det(M))
print("np.linalg.det(M)         =", np.linalg.det(M), "  # 검산용")

v1, v2, v3 = M
print("\n선형종속 관계 : v1 + v2 =", v1 + v2, " == v3 =", v3)
print("세 벡터가 같은 평면 위인지(스칼라 삼중곱) :", dot(v1, cross(v2, v3)))

행 사다리꼴 U =
 [[1. 0. 1.]
 [0. 1. 1.]
 [0. 0. 0.]]

피벗 열 : [0, 1]  -> rank = 2
rank(M) (직접 구현)      = 2
np.linalg.matrix_rank(M) = 2   # 검산용

det(M) (직접 구현)       = 0.0
np.linalg.det(M)         = 0.0   # 검산용

선형종속 관계 : v1 + v2 = [1. 1. 2.]  == v3 = [1. 1. 2.]
세 벡터가 같은 평면 위인지(스칼라 삼중곱) : 0.0


In [13]:
# --- 검증 ---
ok = check("rank 가 2 (3 이 아님)", rank(M) == 2)
ok &= check("직접 구현 rank == np.linalg.matrix_rank", rank(M) == np.linalg.matrix_rank(M))
ok &= check("v3 == v1 + v2 (선형종속)", np.allclose(v3, v1 + v2))
ok &= check("행렬식이 0", np.isclose(det(M), 0.0))
ok &= check("직접 구현 det == np.linalg.det", np.isclose(det(M), np.linalg.det(M)))
ok &= check("rank < 3 과 det == 0 이 서로 일관", (rank(M) < 3) == bool(np.isclose(det(M), 0.0)))
ok &= check("스칼라 삼중곱도 0 (같은 평면)", np.isclose(dot(v1, cross(v2, v3)), 0.0))

# 반례: 선형독립인 세 벡터는 rank 3, det != 0
I3 = np.eye(3)
ok &= check("단위행렬은 rank 3, det 1", rank(I3) == 3 and np.isclose(det(I3), 1.0))
print("\n1-6 전체 통과:", ok)

[PASS] rank 가 2 (3 이 아님)
[PASS] 직접 구현 rank == np.linalg.matrix_rank
[PASS] v3 == v1 + v2 (선형종속)
[PASS] 행렬식이 0
[PASS] 직접 구현 det == np.linalg.det
[PASS] rank < 3 과 det == 0 이 서로 일관
[PASS] 스칼라 삼중곱도 0 (같은 평면)
[PASS] 단위행렬은 rank 3, det 1

1-6 전체 통과: True


## 답안 템플릿 정리

In [14]:
summary = """
1. 내적: {dotv:.0f} / 사이각: {ang:.4f} 도
   - 손계산(24, arccos(0.96) = {hand:.4f} 도)과 일치 여부: True

2. 영벡터 정규화 시 결과: 보호 없이 나누면 [nan nan nan] (RuntimeWarning)
   - 선택한 처리: |v| <= 1e-12 이면 ValueError (fail-fast)
   - 근거: nan 은 예외 없이 전파되고 비교가 항상 False 라 assert 로도 못 잡는다.
           방향이 정의되지 않는 입력이므로 값을 지어내지 않고 호출부에 넘긴다.

3. 정사영 검증: 수직성 True (rej·b = {perp:.1e}) / 합 복원 True

4. skew(a) @ b 와 np.cross(a, b) 일치: True (무작위 200쌍 전수 통과)

5. 세 벡터의 rank: {rk}
   - 3 이 아닌 이유: v3 = v1 + v2 라 세 번째 벡터가 새 방향을 더하지 않는다.
     세 벡터는 3차원 전체가 아니라 평면 하나만 span 한다.
   - 행렬식 값: {dt:.1f}  -> rank < 3 과 det = 0 이 서로 일관
""".format(
    dotv=dot([3.0, 4.0, 0.0], [4.0, 3.0, 0.0]),
    ang=angle_between([3.0, 4.0, 0.0], [4.0, 3.0, 0.0]),
    hand=np.degrees(np.arccos(0.96)),
    perp=dot(reject([2.0, 3.0, 4.0], [1.0, 0.0, 1.0]), [1.0, 0.0, 1.0]),
    rk=rank(M),
    dt=det(M),
)
print(summary)


1. 내적: 24 / 사이각: 16.2602 도
   - 손계산(24, arccos(0.96) = 16.2602 도)과 일치 여부: True

2. 영벡터 정규화 시 결과: 보호 없이 나누면 [nan nan nan] (RuntimeWarning)
   - 선택한 처리: |v| <= 1e-12 이면 ValueError (fail-fast)
   - 근거: nan 은 예외 없이 전파되고 비교가 항상 False 라 assert 로도 못 잡는다.
           방향이 정의되지 않는 입력이므로 값을 지어내지 않고 호출부에 넘긴다.

3. 정사영 검증: 수직성 True (rej·b = 0.0e+00) / 합 복원 True

4. skew(a) @ b 와 np.cross(a, b) 일치: True (무작위 200쌍 전수 통과)

5. 세 벡터의 rank: 2
   - 3 이 아닌 이유: v3 = v1 + v2 라 세 번째 벡터가 새 방향을 더하지 않는다.
     세 벡터는 3차원 전체가 아니라 평면 하나만 span 한다.
   - 행렬식 값: 0.0  -> rank < 3 과 det = 0 이 서로 일관

